# Explore — sales dashboard
Scratch work to pick the charts for `app.py`. Not published.

Q1 · Where does revenue come from?

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/ecommerce_demand_weekly.csv", parse_dates=["date"])
total = df.revenue.sum()
f"{len(df):,} rows · {df.sku.nunique()} SKUs · {df.date.min():%Y-%m-%d} → {df.date.max():%Y-%m-%d} · ${total:,.0f} revenue"

'24,310 rows · 320 SKUs · 2022-07-04 → 2024-12-30 · $44,125,785 revenue'

## By category — value vs volume
Revenue share next to unit share: which categories earn money vs just move boxes.

In [2]:
cat = df.groupby("category").agg(revenue=("revenue", "sum"), units=("net_units", "sum"), avg_price=("price", "mean"))
cat["revenue_share"] = cat.revenue / total
cat["unit_share"] = cat.units / cat.units.sum()
cat = cat.sort_values("revenue", ascending=False)
cat.style.format({"revenue": "${:,.0f}", "units": "{:,.0f}", "avg_price": "${:,.0f}", "revenue_share": "{:.1%}", "unit_share": "{:.1%}"})

,revenue,units,avg_price,revenue_share,unit_share
category,,,,,
electronics,"$15,948,855","92,534",$175,36.1%,9.9%
home_kitchen,"$6,284,055","108,339",$58,14.2%,11.6%
apparel,"$5,479,256","126,538",$44,12.4%,13.5%
sports_outdoor,"$5,468,172","80,958",$68,12.4%,8.6%
grocery,"$3,505,958","241,256",$15,7.9%,25.8%
beauty,"$3,144,912","130,351",$24,7.1%,13.9%
toys_games,"$2,986,961","89,013",$34,6.8%,9.5%
books_media,"$1,307,615","67,586",$19,3.0%,7.2%


In [3]:
px.bar(cat.reset_index().melt(id_vars="category", value_vars=["revenue_share", "unit_share"]),
       x="value", y="category", color="variable", barmode="group", orientation="h",
       title="Revenue share vs unit share by category").update_xaxes(tickformat=".0%")

## By region
Each SKU sells in exactly one region, so region = the SKU's home market.

In [4]:
assert df.groupby("sku").region.nunique().max() == 1
(df.groupby("region").revenue.sum() / total).sort_values(ascending=False).map("{:.1%}".format)

region
North      26.0%
South      22.1%
West       19.0%
East       16.5%
Central    16.4%
Name: revenue, dtype: str

## Concentration — how many SKUs make 80% of revenue?

In [5]:
sku = df.groupby("sku").revenue.sum().sort_values(ascending=False)
cum = (sku.cumsum() / total).reset_index(drop=True)
n80 = int((cum < 0.8).sum()) + 1
fig = px.line(x=cum.index + 1, y=cum.values, labels={"x": "SKUs ranked by revenue", "y": "cumulative share"},
              title=f"{n80} of {len(sku)} SKUs make 80% of revenue")
fig.add_hline(y=0.8, line_dash="dot").update_yaxes(tickformat=".0%")

## Year over year
2022 is a half year, 2024 has 53 Mondays, and the catalog size changes (4–275 active SKUs a week). Compare per week *and* per SKU-week — only the second is like-for-like.

In [6]:
yr = df.groupby(df.date.dt.year).agg(revenue=("revenue", "sum"), weeks=("date", "nunique"), sku_weeks=("sku", "size"))
yr["per_week"] = yr.revenue / yr.weeks
yr["per_sku_week"] = yr.revenue / yr.sku_weeks
yr[["growth_per_week", "growth_per_sku_week"]] = yr[["per_week", "per_sku_week"]].pct_change().values
yr.style.format({"revenue": "${:,.0f}", "per_week": "${:,.0f}", "per_sku_week": "${:,.0f}", "growth_per_week": "{:+.1%}", "growth_per_sku_week": "{:+.1%}"})

,revenue,weeks,sku_weeks,per_week,per_sku_week,growth_per_week,growth_per_sku_week
date,,,,,,,
2022,"$2,804,668",26,1554,"$107,872","$1,805",+nan%,+nan%
2023,"$19,689,042",52,10891,"$378,635","$1,808",+251.0%,+0.2%
2024,"$21,632,075",53,11865,"$408,152","$1,823",+7.8%,+0.8%


### Correction (review, 30 Sept 2026): per SKU-week is still not like-for-like
Products sell little in their 8-week launch ramp and 11-week decline. 2023 was 14.7% launch weeks (catalog filling up); 2024 only 1.2%, but 10.1% declining. So the all-week average compares different lifecycle mixes. Mature weeks only:

In [7]:
mature = df[(df.is_new_launch == 0) & (df.is_declining == 0) & df.date.dt.year.isin([2023, 2024])]
mix = df[df.date.dt.year.isin([2023, 2024])].groupby(df.date.dt.year)[["is_new_launch", "is_declining"]].mean()
m = mature.groupby(mature.date.dt.year).agg(revenue=("revenue", "sum"), sku_weeks=("sku", "size"))
m["per_sku_week"] = m.revenue / m.sku_weeks
print(mix.round(3))  # share of weeks in launch ramp / decline
f"mature revenue per SKU-week, 2024 vs 2023: {m.per_sku_week[2024] / m.per_sku_week[2023] - 1:+.1%}"

      is_new_launch  is_declining
date                             
2023          0.147         0.054
2024          0.012         0.101


'mature revenue per SKU-week, 2024 vs 2023: -1.9%'

---
Q2 · When are the peaks, and how is demand shifting?

Uses `units_sold` (what a manager actually sees). Stockouts only hide ~3.7% of demand; they get their own view.

## Weekly demand — raw total vs per active SKU
SKUs launch and retire, so the raw total also moves with SKU count. Per-SKU strips that out.

In [8]:
wk = df.groupby("date").agg(units=("units_sold", "sum"), active_skus=("sku", "nunique"))
wk["units_per_sku"] = wk.units / wk.active_skus
print(f"active SKUs per week: {wk.active_skus.min()}–{wk.active_skus.max()}")
px.line(wk.reset_index(), x="date", y=["units", "units_per_sku"], facet_row="variable",
        title="Weekly units: total vs per active SKU").update_yaxes(matches=None)

active SKUs per week: 4–275


## Peak weeks

In [9]:
wk.nlargest(8, "units_per_sku")[["units_per_sku", "active_skus"]].round(1)

,units_per_sku,active_skus
date,,
2024-11-25,70.7,172
2023-11-27,69.4,262
2023-11-20,67.8,263
2024-11-18,67.8,175
2024-12-02,66.4,166
2022-11-28,64.6,102
2024-11-11,63.3,175
2024-12-09,63.2,165


## Category × month — seasonal shape
Each category's monthly units per SKU ÷ its own yearly average. 1.0 = normal month, 1.4 = 40% above normal.

In [10]:
m = df.groupby(["category", "month"]).agg(units=("units_sold", "sum"), sku_weeks=("sku", "size"))
m["per_sku"] = m.units / m.sku_weeks
idx = (m.per_sku / m.per_sku.groupby("category").transform("mean")).unstack("month")
px.imshow(idx.round(2), text_auto=True, color_continuous_scale="RdBu_r", color_continuous_midpoint=1,
          aspect="auto", title="Seasonal index by category and month")

In [11]:
idx.agg(["idxmax", "max", "idxmin", "min"], axis=1).rename(columns={"idxmax": "peak_month", "max": "peak", "idxmin": "low_month", "min": "low"})

,peak_month,peak,low_month,low
category,,,,
apparel,11.0,1.675262,7.0,0.677679
beauty,11.0,1.278733,7.0,0.806516
books_media,11.0,1.350725,8.0,0.819946
electronics,11.0,1.569360,7.0,0.771711
grocery,11.0,1.208181,8.0,0.894970
home_kitchen,11.0,1.414841,7.0,0.835625
sports_outdoor,11.0,1.648955,7.0,0.733086
toys_games,11.0,1.935807,7.0,0.657180


## Demand shifts — which categories grew 2023 → 2024?
Per active SKU-week, so launches/retirements don't fake growth.

In [12]:
full = df[df.date.dt.year.isin([2023, 2024])]
g = full.groupby(["category", full.date.dt.year]).agg(units=("units_sold", "sum"), sku_weeks=("sku", "size"))
g = (g.units / g.sku_weeks).unstack()
g["growth"] = g[2024] / g[2023] - 1
g.sort_values("growth", ascending=False).style.format({2023: "{:.1f}", 2024: "{:.1f}", "growth": "{:+.1%}"})

date,2023,2024,growth
category,,,
apparel,53.8,61.7,+14.8%
grocery,77.8,80.8,+3.8%
electronics,34.5,34.7,+0.6%
books_media,23.0,22.1,-4.2%
home_kitchen,40.0,37.9,-5.2%
sports_outdoor,31.5,29.5,-6.3%
toys_games,31.3,28.3,-9.6%
beauty,53.3,45.8,-14.2%


---
Q3 · Do promotions pay off?

A promo is a ~25% price cut. It pays off in revenue only if units rise more than **1 / 0.75 − 1 = 33.3%**.

## Are promos random? (if not, naive comparisons lie)

In [13]:
pd.DataFrame({
    "promo_rate": [df.on_promo.mean()],
    "rate_in_nov": [df[df.month == 11].on_promo.mean()],
    "rate_new_launch": [df[df.is_new_launch == 1].on_promo.mean()],
    "rate_declining": [df[df.is_declining == 1].on_promo.mean()],
    "stockout_on_promo": [df[df.on_promo == 1].stockout.mean()],
    "stockout_off_promo": [df[df.on_promo == 0].stockout.mean()],
}).T.rename(columns={0: "value"}).style.format("{:.1%}")

,value
promo_rate,11.3%
rate_in_nov,10.8%
rate_new_launch,10.7%
rate_declining,11.2%
stockout_on_promo,4.5%
stockout_off_promo,4.3%


Promos run ~11% of weeks, evenly across months and lifecycle stages. Still, compare **within the same SKU and month** so product mix and season can't bias the result.

In [14]:
df["ym"] = df.date.dt.to_period("M")
sku_cat = df.drop_duplicates("sku").set_index("sku").category
# mean per SKU-month, promo vs not; keep only SKU-months that had both
cell = df.groupby(["sku", "ym", "on_promo"])[["units_sold", "price", "revenue"]].mean().unstack("on_promo").dropna()
cell["category"] = sku_cat.reindex(cell.index.get_level_values("sku")).values

def lift(g):
    return pd.Series({k: g[(k, 1)].sum() / g[(k, 0)].sum() - 1 for k in ["units_sold", "price", "revenue"]})

promo = cell.groupby("category").apply(lift).sort_values("revenue", ascending=False)
promo.loc["ALL"] = lift(cell)
print(f"{len(cell):,} SKU-months with both promo and regular weeks")
promo.style.format("{:+.1%}")

2,308 SKU-months with both promo and regular weeks


,units_sold,price,revenue
category,,,
electronics,+57.3%,-25.0%,+17.9%
toys_games,+53.2%,-25.0%,+14.7%
apparel,+38.7%,-25.0%,+4.8%
beauty,+28.3%,-25.0%,-4.0%
sports_outdoor,+22.7%,-25.0%,-8.0%
books_media,+19.4%,-25.0%,-10.2%
home_kitchen,+16.9%,-25.0%,-12.6%
grocery,+8.4%,-25.0%,-18.8%
ALL,+27.5%,-25.0%,+2.8%


In [15]:
p = promo.drop("ALL").reset_index()
fig = px.scatter(p, x="units_sold", y="revenue", text="category", title="Promo unit uplift vs revenue change (break-even at +33% units)")
fig.add_vline(x=1 / 0.75 - 1, line_dash="dot").add_hline(y=0, line_dash="dot")
fig.update_traces(textposition="top center").update_xaxes(tickformat=".0%").update_yaxes(tickformat=".0%")

## Pull-forward check — do sales dip the week after a promo?
If customers just buy earlier, the uplift is partly borrowed from next week.

In [16]:
s = df.sort_values(["sku", "date"]).copy()
s["prev_promo"] = s.groupby("sku").on_promo.shift(1)
regular = s[s.on_promo == 0]
after = regular.groupby(["sku", "ym", "prev_promo"]).units_sold.mean().unstack("prev_promo").dropna()
f"week after promo vs other regular weeks, same SKU-month: {after[1.0].sum() / after[0.0].sum() - 1:+.1%} ({len(after):,} SKU-months)"

'week after promo vs other regular weeks, same SKU-month: -0.6% (2,161 SKU-months)'

**Caveat:** revenue ≠ profit. No cost data here, so a 25% discount that grows revenue can still lose margin.

---
Q6 · Which products are dying and should be dropped?

`is_declining` comes from the data generator — a real business wouldn't have it. So build the rule from **sales alone**, then use the label only to grade the rule.

## What decline looks like in this data

In [17]:
last = df.date.max()
life = df.groupby("sku").agg(last_week=("date", "max"), decline_weeks=("is_declining", "sum"))
retired = life[life.last_week < last]
print(f"retired before the end: {len(retired)} SKUs, all with exactly {retired.decline_weeks.unique()} declining weeks first")
print(f"live at the end: {(life.last_week == last).sum()} SKUs, declining now: {df[(df.date == last) & (df.is_declining == 1)].sku.nunique()}")

retired before the end: 163 SKUs, all with exactly [11] declining weeks first
live at the end: 157 SKUs, declining now: 0


## The rule
1. Deseasonalize: divide units by the category × **week-of-year** factor, so a July dip or the post-Black-Friday drop doesn't look like death. (A monthly factor under-corrects the Nov spike: it flagged 32 SKUs in the last week vs 6 with weekly.)
2. Skip stockout weeks — low sales there mean no stock, not no demand.
3. `ratio` = last 4 weeks ÷ the 12 weeks before them. Launch weeks excluded.

In [18]:
RECENT, BASE = 4, 12
WATCH, DROP = 0.8, 0.6

s = df.sort_values(["sku", "date"]).copy()
s["woy"] = s.date.dt.isocalendar().week.astype(int).clip(upper=52)  # fold week 53 into 52
m = s.groupby(["category", "woy"]).units_sold.mean()
s = s.join((m / m.groupby("category").transform("mean")).rename("season"), on=["category", "woy"])
s["ds"] = (s.units_sold / s.season).where(s.stockout == 0)
g = s.groupby("sku").ds
recent = g.transform(lambda x: x.rolling(RECENT, min_periods=RECENT - 1).mean())
base = g.transform(lambda x: x.shift(RECENT).rolling(BASE, min_periods=BASE // 2).mean())
s["ratio"] = (recent / base).where(s.is_new_launch == 0)
s["tier"] = pd.cut(s.ratio, [0, DROP, WATCH, float("inf")], labels=["drop", "watch", "keep"])

## Grade the rule against the label (whole history)

In [19]:
ev = s[s.ratio.notna()]
def grade(flag):
    y = ev.is_declining == 1
    d = ev[y].assign(f=flag[y].values, wk=ev[y].groupby("sku").cumcount() + 1)
    return pd.Series({"precision": (flag & y).sum() / flag.sum(), "recall": (flag & y).sum() / y.sum(),
                      "dying_skus_caught": d[d.f].sku.nunique() / d.sku.nunique(),
                      "median_week_caught": d[d.f].groupby("sku").wk.min().median()})
pd.DataFrame({"watch (<0.8)": grade(ev.ratio < WATCH), "drop (<0.6)": grade(ev.ratio < DROP)}).round(2)

,watch (<0.8),drop (<0.6)
precision,0.63,0.98
recall,0.69,0.45
dying_skus_caught,1.00,1.00
median_week_caught,4.00,7.00


Every dying SKU gets caught. **Watch** fires ~4 weeks into the 11-week decline but is noisy; **Drop** fires ~7 weeks in and is almost never wrong — leaving ~4 weeks to clear stock.

In [20]:
one = s[s.sku == retired.index[0]]
fig = px.line(one, x="date", y=["ds", "ratio"], facet_row="variable", title=f"{retired.index[0]}: deseasonalized units and ratio before retirement")
fig.update_yaxes(matches=None)

## Today's list (last week in the data)

In [21]:
now = s[s.date == last]
now.tier.value_counts()

tier
keep     150
watch      6
drop       1
Name: count, dtype: int64

In [22]:
now[now.tier != "keep"][["sku", "category", "region", "ratio", "tier"]].sort_values("ratio").round(2)

,sku,category,region,ratio,tier
8029,BOO-1027,books_media,North,0.50,drop
800,APP-1009,apparel,North,0.75,watch
5992,BOO-1000,books_media,North,0.75,watch
23059,TOY-1024,toys_games,South,0.76,watch
10881,ELE-1024,electronics,Central,0.78,watch
23116,TOY-1025,toys_games,West,0.78,watch
10086,ELE-1014,electronics,North,0.79,watch


## Back to Q2 — was the category shift just lifecycle mix?
Re-run 2023 → 2024 growth on *mature* weeks only (no launch ramp, no decline).

In [23]:
mature = s[(s.is_new_launch == 0) & (s.is_declining == 0) & s.date.dt.year.isin([2023, 2024])]
gm = mature.groupby(["category", mature.date.dt.year]).units_sold.mean().unstack()
gm["growth_mature"] = gm[2024] / gm[2023] - 1
gm.sort_values("growth_mature", ascending=False).style.format({2023: "{:.1f}", 2024: "{:.1f}", "growth_mature": "{:+.1%}"})

date,2023,2024,growth_mature
category,,,
apparel,59.4,64.9,+9.3%
grocery,84.0,85.6,+1.9%
electronics,38.6,37.2,-3.5%
home_kitchen,42.7,40.4,-5.4%
books_media,25.2,23.1,-8.5%
toys_games,33.8,30.9,-8.7%
sports_outdoor,35.0,31.1,-11.1%
beauty,60.5,47.5,-21.6%


Shifts hold on mature weeks (beauty −21.6%, apparel +9.3%), so they're real category trends, not lifecycle mix.

---
Past stockouts · How much money did running out of stock cost us?

`lost units = true_demand − units_sold`. Priced at that week's price, minus the category's return rate so it's comparable to `revenue` (which is net of returns).

In a real business `true_demand` is unknown — it has to be estimated. Here the generator gives it to us, so this page shows the *actual* loss.

In [24]:
df["lost_units"] = df.true_demand - df.units_sold
assert (df.lost_units >= 0).all() and not df.loc[df.stockout == 0, "lost_units"].any()  # loss only in stockout weeks
return_rate = df.groupby("category").returns.sum() / df.groupby("category").units_sold.sum()
df["lost_revenue"] = df.lost_units * df.price * (1 - df.category.map(return_rate))
out = df[df.stockout == 1]
f"{len(out):,} stockout weeks ({df.stockout.mean():.1%} of all) · {df.lost_units.sum():,} units · ${df.lost_revenue.sum():,.0f} lost = {df.lost_revenue.sum() / total:.1%} of revenue · a stockout week loses {1 - out.units_sold.sum() / out.true_demand.sum():.0%} of its demand"

'1,056 stockout weeks (4.3% of all) · 40,301 units · $1,705,423 lost = 3.9% of revenue · a stockout week loses 86% of its demand'

## Where is the money lost?

In [25]:
lost = df.groupby("category").agg(stockout_rate=("stockout", "mean"), lost_revenue=("lost_revenue", "sum"), revenue=("revenue", "sum"))
lost["lost_share_of_revenue"] = lost.lost_revenue / lost.revenue
lost.sort_values("lost_revenue", ascending=False).style.format({"stockout_rate": "{:.1%}", "lost_revenue": "${:,.0f}", "revenue": "${:,.0f}", "lost_share_of_revenue": "{:.1%}"})

,stockout_rate,lost_revenue,revenue,lost_share_of_revenue
category,,,,
electronics,4.3%,"$626,386","$15,948,855",3.9%
home_kitchen,4.3%,"$235,344","$6,284,055",3.7%
apparel,4.4%,"$211,892","$5,479,256",3.9%
sports_outdoor,4.2%,"$207,228","$5,468,172",3.8%
grocery,4.4%,"$138,171","$3,505,958",3.9%
toys_games,4.5%,"$119,693","$2,986,961",4.0%
beauty,4.3%,"$116,845","$3,144,912",3.7%
books_media,4.4%,"$49,865","$1,307,615",3.8%


In [26]:
px.bar(lost.sort_values("lost_revenue").reset_index(), x="lost_revenue", y="category", orientation="h",
       title="Revenue lost to stockouts by category").update_xaxes(tickprefix="$")

## Is there a pattern to *when* stockouts happen?

In [27]:
pd.DataFrame({
    "rate_by_month_min": [df.groupby("month").stockout.mean().min()],
    "rate_by_month_max": [df.groupby("month").stockout.mean().max()],
    "rate_on_promo": [df[df.on_promo == 1].stockout.mean()],
    "rate_off_promo": [df[df.on_promo == 0].stockout.mean()],
    "rate_new_launch": [df[df.is_new_launch == 1].stockout.mean()],
    "longest_run_weeks": [out.groupby((df.stockout != df.groupby("sku").stockout.shift()).cumsum()).size().max()],
}).T.rename(columns={0: "value"}).round(3)

,value
rate_by_month_min,0.038
rate_by_month_max,0.052
rate_on_promo,0.045
rate_off_promo,0.043
rate_new_launch,0.041
longest_run_weeks,2.000


**No pattern.** Every category, month, promo state and lifecycle stage runs at ~4–5%, and almost every stockout lasts one week. Stockouts here are random supply hiccups. So the dollar loss follows **price × volume** — electronics loses most because each lost unit is worth $175, not because it runs out more often.

## Worst-hit products

In [28]:
sku_lost = df.groupby(["sku", "category"]).agg(stockout_weeks=("stockout", "sum"), lost_revenue=("lost_revenue", "sum")).reset_index()
share = sku_lost.lost_revenue.sort_values(ascending=False).cumsum() / sku_lost.lost_revenue.sum()
print(f"{int((share < 0.5).sum()) + 1} of {len(sku_lost)} SKUs account for half the loss")
sku_lost.nlargest(10, "lost_revenue").style.format({"lost_revenue": "${:,.0f}"})

54 of 320 SKUs account for half the loss


,sku,category,stockout_weeks,lost_revenue
126,ELE-1006,electronics,6,"$64,045"
151,ELE-1031,electronics,4,"$31,489"
143,ELE-1023,electronics,4,"$31,038"
137,ELE-1017,electronics,4,"$30,013"
122,ELE-1002,electronics,3,"$28,988"
135,ELE-1015,electronics,4,"$28,865"
121,ELE-1001,electronics,5,"$23,248"
120,ELE-1000,electronics,3,"$22,961"
158,ELE-1038,electronics,3,"$22,879"
141,ELE-1021,electronics,4,"$21,649"
